# RL CA1 — Build merged 30-min dataset + EDA

**Inputs** (already in folder):
- `Ausgrid_solar_home_data/Solar home 2012-2013.csv`  (GG=PV, GC=load, CL=controlled)
- `aemo/PRICE_AND_DEMAND_2012*_NSW1.csv`  (run `fetch_aemo.py` first)

**Outputs:**
- `data/merged_30min.csv` — long: one row per customer × 30-min timestamp
- `eda/*.png` — EDA figures

In [ ]:
import re, sys
from pathlib import Path
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

In [ ]:
BASE_PATH = Path.cwd()
AUSGRID_DIR = BASE_PATH / "Ausgrid_solar_home_data"
AUSGRID_FILE = AUSGRID_DIR / "Solar home 2012-2013.csv"
AEMO_DIR = BASE_PATH / "aemo"
DATA_DIR = BASE_PATH / "data"
EDA_DIR = BASE_PATH / "eda"

MARKET_TZ = "Etc/GMT-10"
LOCAL_TZ = "Australia/Sydney"
HH_RE = re.compile(r"^\d{1,2}:\d{2}$")

In [ ]:
def load_ausgrid(n_customers: int) -> pd.DataFrame:
    if not AUSGRID_FILE.exists():
        sys.exit(f"Missing {AUSGRID_FILE}")
    df = pd.read_csv(AUSGRID_FILE, skiprows=1, dtype={"Postcode": str})
    df.columns = [c.strip() for c in df.columns]

    desc = ["Customer", "Generator Capacity", "Postcode",
            "Consumption Category", "date"]
    hh_cols = [c for c in df.columns if HH_RE.match(c)]
    if len(hh_cols) != 48:
        print(f"WARN: expected 48 half-hour cols, found {len(hh_cols)}", file=sys.stderr)

    df = df[df["Consumption Category"].isin(["GG", "GC"])].copy()
    chosen = sorted(df["Customer"].unique())[:n_customers]
    df = df[df["Customer"].isin(chosen)].copy()
    print(f"Ausgrid: {len(chosen)} customers {chosen}, {len(hh_cols)} half-hour cols")

    long = df.melt(id_vars=["Customer", "Consumption Category", "date"],
                   value_vars=hh_cols, var_name="hh", value_name="kwh")
    long["kwh"] = pd.to_numeric(long["kwh"], errors="coerce")

    def hh_to_minutes(label: str) -> int:
        h, m = label.split(":")
        mins = int(h) * 60 + int(m)
        return 1440 if mins == 0 else mins

    long["min_end"] = long["hh"].map(hh_to_minutes)
    base = pd.to_datetime(long["date"], dayfirst=True, errors="coerce")
    long["ts_naive"] = base + pd.to_timedelta(long["min_end"], unit="m")
    long = long.dropna(subset=["ts_naive"])

    ts_local = long["ts_naive"].dt.tz_localize(
        LOCAL_TZ, ambiguous="NaT", nonexistent="shift_forward")
    keep = ts_local.notna()
    dropped = (~keep).sum()
    if dropped:
        print(f"  dropped {dropped} rows at DST transitions (ambiguous local time)")
    long = long[keep].copy()
    long["timestamp"] = ts_local[keep].dt.tz_convert(MARKET_TZ)

    out = (long.pivot_table(index=["Customer", "timestamp"],
                            columns="Consumption Category", values="kwh",
                            aggfunc="first")
                .reset_index()
                .rename(columns={"Customer": "customer",
                                 "GG": "pv_kwh", "GC": "load_kwh"}))
    out.columns.name = None
    for col in ("pv_kwh", "load_kwh"):
        if col not in out.columns:
            out[col] = pd.NA
    return out[["customer", "timestamp", "pv_kwh", "load_kwh"]]

In [ ]:
def load_aemo() -> pd.DataFrame:
    csvs = sorted(AEMO_DIR.glob("PRICE_AND_DEMAND_*_NSW1.csv"))
    if not csvs:
        sys.exit(f"No AEMO CSVs in {AEMO_DIR}. Run: python3 fetch_aemo.py")
    df = pd.concat((pd.read_csv(p) for p in csvs), ignore_index=True)
    df.columns = [c.strip().upper() for c in df.columns]
    ts = pd.to_datetime(df["SETTLEMENTDATE"], errors="coerce")
    df["timestamp"] = ts.dt.tz_localize(MARKET_TZ)
    df["price_aud_per_kwh"] = pd.to_numeric(df["RRP"], errors="coerce") / 1000.0
    df["total_demand_mw"] = pd.to_numeric(df["TOTALDEMAND"], errors="coerce")
    df = df.dropna(subset=["timestamp"]).drop_duplicates("timestamp")
    print(f"AEMO: {len(df):,} price rows "
          f"({df['timestamp'].min()} -> {df['timestamp'].max()})")
    return df[["timestamp", "price_aud_per_kwh", "total_demand_mw"]]

In [ ]:
def merge(ausgrid: pd.DataFrame, aemo: pd.DataFrame) -> pd.DataFrame:
    m = (ausgrid.merge(aemo, on="timestamp", how="inner")
                .sort_values(["customer", "timestamp"]).reset_index(drop=True))
    m["net_load_kwh"] = m["load_kwh"] - m["pv_kwh"]
    print(f"Merged: {len(m):,} rows, {m['customer'].nunique()} customers, "
          f"{m['timestamp'].dt.date.nunique()} days")
    return m

In [ ]:
def eda(m: pd.DataFrame) -> None:
    EDA_DIR.mkdir(parents=True, exist_ok=True)
    one = m[m["customer"] == m["customer"].iloc[0]].copy()
    one["hour"] = one["timestamp"].dt.hour + one["timestamp"].dt.minute / 60.0

    prof = one.groupby("hour")[["pv_kwh", "load_kwh"]].mean()
    price_prof = one.groupby("hour")["price_aud_per_kwh"].mean()
    fig, ax1 = plt.subplots(figsize=(9, 5))
    prof.plot(ax=ax1); ax1.set_ylabel("kWh / 30-min"); ax1.set_xlabel("hour")
    ax2 = ax1.twinx(); price_prof.plot(ax=ax2, color="black", linestyle="--")
    ax2.set_ylabel("price (AUD/kWh)")
    ax1.set_title(f"Avg daily profile — customer {one['customer'].iloc[0]}")
    fig.tight_layout(); fig.savefig(EDA_DIR / "daily_profile.png", dpi=120); plt.close(fig)

    fig, ax = plt.subplots(figsize=(8, 4))
    m["price_aud_per_kwh"].plot(kind="hist", bins=80, ax=ax)
    ax.set_xlabel("price (AUD/kWh)"); ax.set_title("NSW1 spot price (2012-13)")
    fig.tight_layout(); fig.savefig(EDA_DIR / "price_hist.png", dpi=120); plt.close(fig)

    fig, ax = plt.subplots(figsize=(7, 6))
    s = m.sample(min(len(m), 5000), random_state=0)
    ax.scatter(s["price_aud_per_kwh"], s["net_load_kwh"], s=4, alpha=0.3)
    ax.set_xlabel("price (AUD/kWh)"); ax.set_ylabel("net load (kWh)")
    ax.set_title("Net load vs price")
    fig.tight_layout(); fig.savefig(EDA_DIR / "netload_vs_price.png", dpi=120); plt.close(fig)
    print(f"EDA -> {EDA_DIR}")

## Configuration

Set `N_CUSTOMERS` and toggle EDA below.

In [ ]:
N_CUSTOMERS = 5
NO_EDA = False

In [ ]:
DATA_DIR.mkdir(parents=True, exist_ok=True)
ausgrid = load_ausgrid(N_CUSTOMERS)
aemo = load_aemo()
merged = merge(ausgrid, aemo)
out = DATA_DIR / "merged_30min.csv"
merged.to_csv(out, index=False)
print(f"Wrote {out}")

In [ ]:
cov = merged.groupby("customer")["timestamp"].agg(["min", "max", "count"])
print("Coverage per customer:\n", cov)
print("\nMissing fraction:\n",
      merged[["pv_kwh", "load_kwh", "price_aud_per_kwh"]].isna().mean())

In [ ]:
if not NO_EDA:
    eda(merged)